## Instalação da API

In [ ]:
#%pip install statsbombpy

## Importações

In [ ]:
from statsbombpy import sb
import pandas as pd

from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from IPython.display import display
from PIL import Image
import json
import os
import matplotlib.pyplot as plt
import numpy as np
from math import pi
from pandas import json_normalize
import plotly.graph_objects as go

# Dataset

- Quatro conjuntos:
    - Competições
    - Partidas
    - Jogadores convocados para um partida
    - Eventos de um partida

### Competições

In [ ]:
sb.competitions()

### Partidas (Bundesliga 23/24)

In [ ]:
sb.matches(competition_id=9, season_id=281)

In [ ]:
sb.matches(competition_id=9, season_id=281).columns

### Jogadores convocados (partida Bayer Leverkusen - Werder Bremen)

In [ ]:
sb.lineups(match_id=3895302)["Bayer Leverkusen"]

In [ ]:
sb.lineups(match_id=3895302)["Bayer Leverkusen"].columns

### Eventos (partida Bayer Leverkusen - Werder Bremen)

In [ ]:
sb.events(match_id=3895302)

In [ ]:
# variáveis presentes
eventos = sb.events(match_id=3895302)
list(eventos.keys())

In [ ]:
#pd.set_option('display.max_colwidth', None)
sb.events(match_id=3895302, split=True, flatten_attrs=False)["shots"]

- Cada tipo na variável "type" tem variáveis adiconais. Vamos ver para o exemplo do tipo "remate"

In [ ]:
df = sb.events(match_id=3895302, split=True, flatten_attrs=False)["shots"]
print(df["shot"].iloc[0].keys())

# EDA

## Competições

In [ ]:
profile = ProfileReport(sb.competitions(), explorative=True)
profile.to_notebook_iframe()

## Partidas

In [ ]:
profile = ProfileReport(sb.matches(competition_id=9, season_id=281), explorative=True)
profile.to_notebook_iframe()

In [ ]:
matches_df = sb.matches(competition_id=9, season_id=281)
print(matches_df.isnull().sum()) 

In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = sb.matches(competition_id=9, season_id=281)
team_name = "Bayer Leverkusen"

home_matches = df[df["home_team"] == team_name]
away_matches = df[df["away_team"] == team_name]

# golos marcados
home_goals_scored = home_matches["home_score"].sum()
away_goals_scored = away_matches["away_score"].sum()
total_goals_scored = home_goals_scored + away_goals_scored

# golos sofridos
home_goals_conceded = home_matches["away_score"].sum()
away_goals_conceded = away_matches["home_score"].sum()
total_goals_conceded = home_goals_conceded + away_goals_conceded

default_colors = ["#636EFA", "#EF553B"] 

# golos marcados
fig1 = go.Figure()
fig1.add_trace(go.Pie(
    labels=["Home", "Away"],
    values=[home_goals_scored, away_goals_scored],
    textinfo="label+percent",
    hole=0.4,
    marker=dict(colors=default_colors) 
))
fig1.add_annotation(
    text=f"Total: {total_goals_scored}", 
    x=0.5, y=0.5, 
    showarrow=False, font_size=16
)
fig1.update_layout(title_text="Goals Scored by Bayer Leverkusen")

# golos sofridos
fig2 = go.Figure()
fig2.add_trace(go.Pie(
    labels=["Home", "Away"],
    values=[home_goals_conceded, away_goals_conceded],
    textinfo="label+percent",
    hole=0.4,
    marker=dict(colors=default_colors) 
))
fig2.add_annotation(
    text=f"Total: {total_goals_conceded}", 
    x=0.5, y=0.5, 
    showarrow=False, font_size=16
)
fig2.update_layout(title_text="Goals Conceded by Bayer Leverkusen")

fig1.show()
fig2.show()


## Jogadores Convocados

In [ ]:
profile = ProfileReport(sb.lineups(match_id=3895302)["Bayer Leverkusen"], explorative=True)
profile.to_notebook_iframe()

In [ ]:
df_matches = sb.matches(competition_id=9, season_id=281)
player_name = "Florian Wirtz"
team_name = "Bayer Leverkusen"
total_called_up = 0  # Convocado
total_played = 0      # Jogou
total_started = 0     # Titular
for match_id in df_matches["match_id"]:
    lineup = sb.lineups(match_id)[team_name]
    player_data = lineup[lineup["player_name"] == player_name]
    if not player_data.empty:
        total_called_up += 1
        positions = player_data["positions"].values[0]  
        if positions: 
            total_played += 1
            for position in positions:
                if position.get("start_reason") == "Starting XI":
                    total_started += 1
                    break 
print(f"Total de jogos convocado: {total_called_up}")
print(f"Total de jogos que jogou: {total_played}")
print(f"Total de jogos como titular: {total_started}")

In [ ]:
total_team_matches = len(df_matches)
total_called_up = 32 
total_played = 32  
total_started = 26 
total_substitute = total_called_up - total_started

default_colors = ["#636EFA", "#EF553B"] 
categories = ["Total Matches vs Called Up", "Called Up vs Played", "Starter vs Substitute"]
higher_values = [total_team_matches, total_called_up, total_started]
lower_values = [total_called_up, total_played, total_substitute]

fig_pie1 = go.Figure()
fig_pie1.add_trace(go.Pie(
    labels=["Called Up", "Not Called Up"],
    values=[total_called_up, total_team_matches - total_called_up],
    hole=0.4,
    textinfo="label+percent",
    marker=dict(colors=default_colors)
))
fig_pie1.update_layout(title_text="Percentage of Matches Called Up")

fig_pie2 = go.Figure()
fig_pie2.add_trace(go.Pie(
    labels=["Played", "Did Not Play"],
    values=[total_played, total_called_up - total_played],
    hole=0.4,
    textinfo="label+percent",
    marker=dict(colors=default_colors)
))
fig_pie2.update_layout(title_text="Games Played vs Called Up")

fig_pie3 = go.Figure()
fig_pie3.add_trace(go.Pie(
    labels=["Starter", "Substitute"],
    values=[total_started, total_substitute],
    hole=0.4,
    textinfo="label+percent",
    marker=dict(colors=default_colors)
))
fig_pie3.update_layout(title_text="Starter vs Substitute")

fig_pie1.show()
fig_pie2.show()
fig_pie3.show()


### Eventos

In [ ]:
profile = ProfileReport(sb.events(match_id=3895302), explorative=True)
profile.to_notebook_iframe()

In [ ]:
import pandas as pd
from collections import Counter
import plotly.express as px

# Obter todos os jogos da Bundesliga
df_matches = sb.matches(competition_id=9, season_id=281)

# Definir nome da equipa e do jogador
team_name = "Bayer Leverkusen"
player_name = "Florian Wirtz"

# Inicializar contadores
total_team_events = 0
total_player_events = 0
player_event_types = Counter()  # Contador para os tipos de eventos do jogador

# Percorrer cada jogo e analisar eventos
for match_id in df_matches["match_id"]:
    df_events = sb.events(match_id)  # Obter eventos da partida
    
    # Filtrar eventos da equipa Bayer Leverkusen
    team_events = df_events[df_events["team"] == team_name]
    total_team_events += len(team_events)
    
    # Filtrar eventos do jogador Florian Wirtz
    player_events = df_events[df_events["player"] == player_name]
    total_player_events += len(player_events)
    
    # Contabilizar os tipos de eventos do jogador
    player_event_types.update(player_events["type"])

# Converter a contagem de eventos do jogador em um DataFrame
df_player_events = pd.DataFrame(player_event_types.items(), columns=["Event Type", "Frequency"])

# **Ordenar os eventos do mais frequente para o menos frequente**
df_player_events = df_player_events.sort_values(by="Frequency", ascending=False)

# Mostrar resultados
print(f"Total events of the team {team_name}: {total_team_events}")
print(f"Total events of the player {player_name}: {total_player_events}")
print("\nFrequency of event types of the player Florian Wirtz:")
print(df_player_events)

# Criar gráfico de barras ordenado
fig = px.bar(df_player_events, 
             x="Event Type", 
             y="Frequency", 
             title=f"Event Frequency of {player_name}",
             labels={"Event Type": "Event Type", "Frequency": "Number of Events"},
             text_auto=True, 
             color="Frequency")

# Ajustar layout para melhor visualização
fig.update_layout(xaxis={'categoryorder':'total descending'})

# Mostrar gráfico
fig.show()


In [ ]:
sb.events(match_id=3895302)["type"].unique()

In [ ]:
sb.events(match_id=3930174, split=True, flatten_attrs=False)["shots"].columns

In [ ]:
sb.events(match_id=3930174, split=True, flatten_attrs=False)["passes"].columns

---------------------------------

#  Visualizações

---------------------------------

# Apresentação Jogador

In [ ]:
def convert_to_per_90(value, minutes_played):
    return round((value / minutes_played) * 90, 2) if minutes_played > 0 else 0

def safe_divide(numerator, denominator):
    return round((numerator / denominator) * 100, 2) if denominator else 0

def parse_time(t):
    if t is None or t == 'Final Whistle':
        return 90.0
    mins, secs = map(int, t.split(':'))
    return mins + secs / 60

DATA_PATH = './data'

competition_name = "1. Bundesliga"
season_name = "2023/2024"
goals = 0
assists = 0

with open(os.path.join(DATA_PATH, 'competitions.json'), encoding='utf-8') as f:
    competitions = json.load(f)

comp_id, season_id = None, None
for c in competitions:
    if c['competition_name'] == competition_name and c['season_name'] == season_name:
        comp_id = c['competition_id']
        season_id = c['season_id']
assert comp_id and season_id, "Competição não encontrada."

matches_path = os.path.join(DATA_PATH, 'matches', str(comp_id), f'{season_id}.json')
with open(matches_path, encoding='utf-8') as f:
    matches = json.load(f)
match_ids = [m['match_id'] for m in matches]

player_name = "Florian Wirtz"

minutes_played = 0

for match_id in match_ids:
    try:
        lineup_path = os.path.join(DATA_PATH, 'lineups', f'{match_id}.json')
        with open(lineup_path, encoding='utf-8') as f:
            lineup = json.load(f)

        for team in lineup:
            for player in team['lineup']:
                if player.get('player_name') == player_name:
                    for pos in player['positions']:
                        start = parse_time(pos.get("from"))
                        end = parse_time(pos.get("to"))
                        delta = max(0, end - start)
                        minutes_played += delta

        events_path = os.path.join(DATA_PATH, 'events', f'{match_id}.json')
        with open(events_path, encoding='utf-8') as f:
            events = json.load(f)

        df = json_normalize(events, sep='_')
        df_player = df[df['player_name'] == player_name]

        if df_player.empty:
            continue

        # Finalização
        shots = df_player[df_player['type_name'] == 'Shot']
        goals += shots['shot_outcome_name'].isin(['Goal']).sum()

        # Passes
        passes = df_player[df_player['type_name'] == 'Pass']
        assists += passes['pass_goal_assist'].fillna(False).astype(bool).sum()  

    except Exception as e:
        continue
        
df = sb.lineups(match_id=3895302)["Bayer Leverkusen"]

wirtz = df[df['player_name'] == "Florian Wirtz"].iloc[0]

nacionallity = wirtz['country']
position = wirtz['positions'][-1]['position'] if wirtz['positions'] else None


df_matches = sb.matches(competition_id=9, season_id=281)
player_name = "Florian Wirtz"
team_name = "Bayer Leverkusen"
total_called_up = 0  # Convocado
total_played = 0      # Jogou
total_started = 0     # Titular
for match_id in df_matches["match_id"]:
    lineup = sb.lineups(match_id)[team_name]
    player_data = lineup[lineup["player_name"] == player_name]
    if not player_data.empty:
        total_called_up += 1
        positions = player_data["positions"].values[0]  
        if positions: 
            total_played += 1
            for position in positions:
                if position.get("start_reason") == "Starting XI":
                    total_started += 1
                    break 
                    

# Carregar imagem
img = Image.open("Wirtz.jpg")  

# Nome do jogador
nome_jogador = "Florian Wirtz"

# Dados do jogador
dados_esquerda = {
    "21 years": "Age",
    wirtz['country']: "Nationality",
    "Bayer Leverkusen": "Team",
    wirtz['positions'][-1]['position']: "Position"
}

dados_direita = {
    total_played: "Matches",
    round(minutes_played): "Minutes Played",
    goals: "Goals",
    assists: "Assists"
}

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[0.505],  
    y=[0.94],  
    text=f"<b><span style='font-size:32px;'>{nome_jogador}</span></b>",
    mode="text",
    textfont=dict(size=32, color="black", family="Helvetica"),
    showlegend=False
))

image_width = 0.75
image_height = 0.7

fig.add_layout_image(
    dict(
        source=img,
        xref="paper",
        yref="paper",
        x=0.5,
        y=0.47,
        sizex=image_width,
        sizey=image_height,
        xanchor="center",
        yanchor="middle",
        sizing="contain",
        opacity=1,
        layer="below"
    )
)

y_positions = np.linspace(0.73, 0.2, len(dados_esquerda))

# Dados do lado esquerdo
for (valor, chave), y in zip(dados_esquerda.items(), y_positions):
    fig.add_trace(go.Scatter(
        x=[0.13],  
        y=[y], 
        text=f"<b><span style='font-size:18px;'>{valor}</span></b><br><span style='font-size:17px;'>{chave}</span>",
        mode="text",
        textfont=dict(size=18, color="black", family="Helvetica"),  
        showlegend=False
    ))

# Dados do lado direito
for (valor, chave), y in zip(dados_direita.items(), y_positions):
    fig.add_trace(go.Scatter(
        x=[0.88],  
        y=[y], 
        text=f"<b><span style='font-size:18px;'>{valor}</span></b><br><span style='font-size:17px;'>{chave}</span>",
        mode="text",
        textfont=dict(size=18, color="black", family="Helvetica"),
        showlegend=False
    ))

# Layout
fig.update_layout(
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0, 1]),
    yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0, 1]),
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=0, r=0, t=0, b=0),  
    width=900,
    height=500
)
fig.show()

# Radar

**Dados do jogador**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from pandas import json_normalize

def convert_to_per_90(value, minutes_played):
    return round((value / minutes_played) * 90, 2) if minutes_played > 0 else 0

def parse_time(t):
    if t is None or t == 'Final Whistle':
        return 90.0
    mins, secs = map(int, t.split(':'))
    return mins + secs / 60

def load_player_stats(player_name, comp_id, season_id):
    DATA_PATH = './data'

    matches_path = os.path.join(DATA_PATH, 'matches', str(comp_id), f'{season_id}.json')
    with open(matches_path, encoding='utf-8') as f:
        matches = json.load(f)
    match_ids = [m['match_id'] for m in matches]

    stats = {
        'Goals': 0, 'Shots': 0, 'Shots on Target': 0, 'xG': 0, 'Assists': 0, 'Key Passes': 0,
        'Completed Passes': 0, 'Progressive Passes': 0, 'Successful Dribbles': 0,
        'Pressures': 0, 'Ball Recoveries': 0, 'Tackles': 0,
    }
    minutes_played = 0

    for match_id in match_ids:
        try:
            lineup_path = os.path.join(DATA_PATH, 'lineups', f'{match_id}.json')
            with open(lineup_path, encoding='utf-8') as f:
                lineup = json.load(f)

            for team in lineup:
                for player in team['lineup']:
                    if player.get('player_name') == player_name:
                        for pos in player['positions']:
                            start = parse_time(pos.get("from"))
                            end = parse_time(pos.get("to"))
                            delta = max(0, end - start)
                            minutes_played += delta

            events_path = os.path.join(DATA_PATH, 'events', f'{match_id}.json')
            with open(events_path, encoding='utf-8') as f:
                events = json.load(f)

            df = json_normalize(events, sep='_')
            df_player = df[df['player_name'] == player_name]

            if df_player.empty:
                continue

            shots = df_player[df_player['type_name'] == 'Shot']
            stats['Shots'] += len(shots)
            stats['Shots on Target'] += shots['shot_outcome_name'].isin(['Goal', 'Saved', 'Saved to Post']).sum()
            stats['Goals'] += (shots['shot_outcome_name'] == 'Goal').sum()
            stats['xG'] += shots['shot_statsbomb_xg'].sum()

            passes = df_player[df_player['type_name'] == 'Pass']
            stats['Assists'] += passes['pass_goal_assist'].fillna(False).astype(bool).sum()
            stats['Key Passes'] += passes['pass_shot_assist'].fillna(False).astype(bool).sum()
            stats['Progressive Passes'] += passes['pass_length'].gt(25).sum()
            stats['Completed Passes'] += passes['pass_outcome_name'].isna().sum()

            dribbles = df_player[df_player['type_name'] == 'Dribble']
            stats['Successful Dribbles'] += (dribbles['dribble_outcome_name'] == 'Complete').sum()

            stats['Pressures'] += (df_player['type_name'] == 'Pressure').sum()
            stats['Ball Recoveries'] += (df_player['type_name'] == 'Ball Recovery').sum()
            stats['Tackles'] += ((df_player['type_name'] == 'Duel') & (df_player['duel_type_name'] == 'Tackle')).sum()

        except Exception as e:
            continue

    per90 = {k: convert_to_per_90(v, minutes_played) for k, v in stats.items()}

    values = [
        per90['Goals'], per90['Shots'], per90['xG'], per90['Completed Passes'],
        per90['Progressive Passes'], per90['Assists'], per90['Key Passes'],
        per90['Successful Dribbles'], per90['Pressures'], per90['Ball Recoveries'], per90['Tackles']
    ]

    return values


**Radar**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def _invert(x, limits):
    return limits[1] - (x - limits[0])

class ComplexRadar():
    def __init__(self, fig, variables, ranges):
        self.variables = variables
        self.ranges = ranges
        self.n = len(variables)
        self.angles = np.linspace(0, 2 * np.pi, self.n, endpoint=False).tolist()
        self.angles += self.angles[:1]

        self.ax = fig.add_axes([0.13, 0.05, 0.7, 0.7], polar=True)
        self.ax.set_facecolor("#F9FAFC")
        self.ax.spines["polar"].set_color("#CCCCCC")
        self.ax.set_theta_offset(np.pi / 2)
        self.ax.set_theta_direction(-1)
        self.ax.set_rgrids([])
        self.ax.set_yticklabels([])
        self.ax.set_xticks([])
        self.ax.set_ylim(0, 1)

        for angle in self.angles[:-1]:
            self.ax.plot([angle, angle], [0, 1], color="#DDDDDD", linewidth=1)

        for i, label in enumerate(variables):
            angle = self.angles[i]
            label_radius = 1.08
            rotation = np.degrees(angle)

            ha = 'center'
            va = 'center'
            if 0 <= rotation < 90 or 270 < rotation <= 360:
                va = "bottom"
            elif 90 < rotation < 270:
                va = "top"
            if 45 < rotation < 135:
                ha = "left"
            elif 225 < rotation < 315:
                ha = "right"

            self.ax.text(angle, label_radius, label, size=10, ha=ha, va=va,
                         fontweight='bold', color='#222222')

    def scale_data(self, data, ranges):
        scaled = []
        for d, (y1, y2) in zip(data, ranges):
            if y1 > y2:
                d = _invert(d, (y1, y2))
                y1, y2 = y2, y1
            scaled.append((d - y1) / (y2 - y1))
        return scaled

    def plot_compare(self, data1, data2, color1='#B40605', color2='#0E2FAA'):
        sdata1 = self.scale_data(data1, self.ranges)
        sdata2 = self.scale_data(data2, self.ranges)
        sdata1 += sdata1[:1]
        sdata2 += sdata2[:1]

        self.ax.plot(self.angles, sdata1, color=color1, lw=2.5, zorder=3, marker='o', markersize=6)
        self.ax.plot(self.angles, sdata2, color=color2, lw=2.5, zorder=3, marker='o', markersize=6)

        self.ax.fill(self.angles, sdata1, color=color1, alpha=0.3, zorder=2)
        self.ax.fill(self.angles, sdata2, color=color2, alpha=0.3, zorder=2)

        for i in range(self.n):
            angle = self.angles[i]
            r1 = sdata1[i]
            r2 = sdata2[i]
            offset = 0.09  # proximadade dos numeros 
            if data1[i] >= data2[i]:
                self.ax.text(angle, r1 + offset, f"{data1[i]:.1f}", fontsize=9, ha='center', va='center', color=color1)
                self.ax.text(angle, r2 - offset, f"{data2[i]:.1f}", fontsize=9, ha='center', va='center', color=color2)
            else:
                self.ax.text(angle, r1 - offset, f"{data1[i]:.1f}", fontsize=9, ha='center', va='center', color=color1)
                self.ax.text(angle, r2 + offset, f"{data2[i]:.1f}", fontsize=9, ha='center', va='center', color=color2)

    def plot_single(self, data, color='#B40605'):
        sdata = self.scale_data(data, self.ranges)
        sdata += sdata[:1]

        self.ax.plot(self.angles, sdata, color=color, lw=2.5, zorder=3, marker='o', markersize=6)
        self.ax.fill(self.angles, sdata, color=color, alpha=0.3, zorder=2)

        for i in range(self.n):
            angle = self.angles[i]
            r = sdata[i]
            self.ax.text(angle, r + 0.10, f"{data[i]:.1f}", fontsize=9, ha='center', va='center', color=color)


            
# RADAR INDIVIDUAL
def plot_single_player_radar(labels, ranges, stats, player_name, comp_label, save_path=None):
    fig = plt.figure(figsize=(10, 9))
    radar = ComplexRadar(fig, labels, ranges)
    radar.plot_single(stats, color='#B40605')
    
    fig.subplots_adjust(left=0.1, right=0.9, top=0.85, bottom=0.08)
    
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    else:
        plt.show()
    
    

# RADAR COMPARAÇÃO
def plot_compare_players_with_images(labels, ranges, stats1, stats2,
                                     player1_name, player2_name,
                                     comp1_label="", comp2_label="",
                                     image1_path=None, image2_path=None, save_path=None):
    fig = plt.figure(figsize=(10, 9))
    radar = ComplexRadar(fig, labels, ranges)
    radar.plot_compare(stats1, stats2)

    if image1_path:
        ax_img1 = fig.add_axes([0.02, 0.89, 0.11, 0.11], zorder=-1)
        img1 = mpimg.imread(image1_path)
        ax_img1.imshow(img1)
        ax_img1.axis('off')
        fig.text(0.073, 0.855, player1_name, ha='center', fontsize=17, fontweight='bold', color='#B40605')
        fig.text(0.073, 0.825, comp1_label, ha='center', fontsize=11, color='#444444')

    if image2_path:
        ax_img2 = fig.add_axes([0.87, 0.89, 0.11, 0.11], zorder=-1)
        img2 = mpimg.imread(image2_path)
        ax_img2.imshow(img2)
        ax_img2.axis('off')
        fig.text(0.926, 0.855, player2_name, ha='center', fontsize=17, fontweight='bold', color='#0E2FAA')
        fig.text(0.926, 0.825, comp2_label, ha='center', fontsize=11, color='#444444')

    fig.subplots_adjust(left=0.1, right=0.9, top=0.85, bottom=0.08)
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    else:
        plt.show()
    
labels = ["Goals", "Shots", "Expected Goals (xG)", "Completed Passes", "Progressive Passes",
          "Assists", "Key Passes", "Successful Dribbles",
          "Pressures", "Ball Recoveries", "Tackles"]

ranges = [(0, 1.2), (0, 6.0), (0, 1.2), (0, 65), (0, 13),
          (0, 0.9), (0, 3.5), (0, 5), (0, 30), (0, 12), (0, 2.5)]



In [ ]:
comp_id_wirtz = 9
season_id_wirtz = 281
player1_name = "Florian Wirtz"


player1_stats = load_player_stats(player1_name, comp_id_wirtz, season_id_wirtz)

plot_single_player_radar(
    labels, ranges,
    stats=player1_stats,
    player_name="Florian Wirtz",
    comp_label="1. Bundesliga 2023/24",
    save_path="radar_wirtz.png"
)

In [ ]:
# Dados Griezmann
comp_id_griezmann = 11
season_id_griezmann = 90
player2_name = "Antoine Griezmann"

player2_stats = load_player_stats(player2_name, comp_id_griezmann, season_id_griezmann)


# Chamada ao radar customizado
plot_compare_players_with_images(
    labels, ranges,
    player1_stats, player2_stats,
    player1_name="Florian Wirtz",
    player2_name="Griezmann",
    comp1_label="1. Bundesliga 2023/24",
    comp2_label="La Liga 2021/22",
    image1_path="cara_wirtz.jpg",
    image2_path="cara_griezmann.png",
    save_path="wirtz_griezmann.png"
)


In [ ]:
# Dados Neymar
comp_id_neymar = 7
season_id_neymar = 235
player2_name = "Neymar da Silva Santos Junior"

player2_stats = load_player_stats(player2_name, comp_id_neymar, season_id_neymar)

plot_compare_players_with_images(
    labels, ranges,
    player1_stats, player2_stats,
    player1_name="Florian Wirtz",
    player2_name="Neymar Junior",
    comp1_label="1. Bundesliga 2023/24",
    comp2_label="Ligue 1 2022/23",
    image1_path="cara_wirtz.jpg",
    image2_path="cara_neymar.webp",
    save_path="wirtz_neymar.png"
)


In [ ]:
# Dados Boniface
comp_id_boniface = 9
season_id_boniface = 281
player2_name = "Victor Okoh Boniface"

player2_stats = load_player_stats(player2_name, comp_id_boniface, season_id_boniface)

plot_compare_players_with_images(
    labels, ranges,
    player1_stats, player2_stats,
    player1_name="Florian Wirtz",
    player2_name="Victor Boniface",
    comp1_label="1. Bundesliga 2023/24",
    comp2_label="1. Bundesliga 2023/24",
    image1_path="cara_wirtz.jpg",
    image2_path="cara_boniface.jpg",
    save_path="wirtz_boniface.png"
)


In [ ]:
# Dados Messi
comp_id_messi = 11
season_id_messi = 90
player2_name = "Lionel Andrés Messi Cuccittini"

player2_stats = load_player_stats(player2_name, comp_id_messi, season_id_messi)

plot_compare_players_with_images(
    labels, ranges,
    player1_stats, player2_stats,
    player1_name="Florian Wirtz",
    player2_name="Lionel Messi",
    comp1_label="1. Bundesliga 2023/24",
    comp2_label="La Liga 2021/22",
    image1_path="cara_wirtz.jpg",
    image2_path="cara_messi.webp",
    save_path="wirtz_messi.png"
)


In [ ]:
# Dados Vitinha
comp_id_vitinha = 7
season_id_vitinha = 235
player2_name = "Kylian Mbappé Lottin"

player2_stats = load_player_stats(player2_name, comp_id_vitinha, season_id_vitinha)

plot_compare_players_with_images(
    labels, ranges,
    player1_stats, player2_stats,
    player1_name="Florian Wirtz",
    player2_name="Kylian Mbappé",
    comp1_label="1. Bundesliga 2023/24",
    comp2_label="Ligue 1 2022/23",
    image1_path="cara_wirtz.jpg",
    image2_path="cara_mbappe.png",
    save_path="wirtz_mbappe.png"
)


In [ ]:
# Dados Grimaldo
comp_id_boniface = 9
season_id_boniface = 281
player2_name = "Amine Adli"

player2_stats = load_player_stats(player2_name, comp_id_boniface, season_id_boniface)

plot_compare_players_with_images(
    labels, ranges,
    player1_stats, player2_stats,
    player1_name="Florian Wirtz",
    player2_name="Amine Adli",
    comp1_label="1. Bundesliga 2023/24",
    comp2_label="1. Bundesliga 2023/24",
    image1_path="cara_wirtz.jpg",
    image2_path="cara_adli.jpg",
    save_path="wirtz_adli.png"
)


In [ ]:
# Dados dembele
comp_id_messi = 11
season_id_messi = 4
player2_name = "Ousmane Dembélé"

player2_stats = load_player_stats(player2_name, comp_id_messi, season_id_messi)

plot_compare_players_with_images(
    labels, ranges,
    player1_stats, player2_stats,
    player1_name="Florian Wirtz",
    player2_name="Ousmane Dembélé",
    comp1_label="1. Bundesliga 2023/24",
    comp2_label="La Liga 2018/19",
    image1_path="cara_wirtz.jpg",
    image2_path="cara_dembele.webp",
    save_path="wirtz_dembele.png"
)


# Campo Matplotlib

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Circle

def draw_pitch(ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))

    # Contorno do campo
    ax.plot([0, 0], [0, 80], color='black', linewidth=4)        # Lateral esquerda
    ax.plot([0, 120], [80, 80], color='black', linewidth=4)     # Linha superior
    ax.plot([120, 120], [80, 0], color='black', linewidth=4)    # Lateral direita
    ax.plot([120, 0], [0, 0], color='black', linewidth=4)       # Linha inferior

    # Linha de meio campo
    ax.plot([60, 60], [0, 80], color='black', linewidth=2)

    # Círculo central
    ax.add_patch(Circle((60, 40), 9.15, color='black', fill=False, linewidth=2))
    ax.add_patch(Circle((60, 40), 0.3, color='black'))  # marca do centro

    # Áreas grandes
    ax.plot([0, 18], [18, 18], color='black', linewidth=2)
    ax.plot([18, 18], [18, 62], color='black', linewidth=2)
    ax.plot([0, 18], [62, 62], color='black', linewidth=2)

    ax.plot([120, 102], [18, 18], color='black', linewidth=2)
    ax.plot([102, 102], [18, 62], color='black', linewidth=2)
    ax.plot([102, 120], [62, 62], color='black', linewidth=2)

    # Arcos da área
    ax.add_patch(Arc((12, 40), width=18.3, height=18.3, angle=0, theta1=310, theta2=50, color='black', linewidth=2))
    ax.add_patch(Arc((108, 40), width=18.3, height=18.3, angle=180, theta1=310, theta2=50, color='black', linewidth=2))

    # Áreas pequenas
    ax.plot([0, 6], [30, 30], color='black', linewidth=2)
    ax.plot([6, 6], [30, 50], color='black', linewidth=2)
    ax.plot([6, 0], [50, 50], color='black', linewidth=2)

    ax.plot([120, 114], [30, 30], color='black', linewidth=2)
    ax.plot([114, 114], [30, 50], color='black', linewidth=2)
    ax.plot([114, 120], [50, 50], color='black', linewidth=2)

    # Marcas de penálti
    ax.add_patch(Circle((12, 40), radius=0.3, color='black', linewidth=2))
    ax.add_patch(Circle((108, 40), radius=0.3, color='black', linewidth=2))

    # Balizas
    ax.plot([0, 0], [36, 44], color='black', linewidth=8)
    ax.plot([120, 120], [36, 44], color='black', linewidth=8)

    # Configurações do gráfico
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)
    ax.set_aspect('equal')
    ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # direção ataque
    ax.annotate(
        '', xy=(68, -3), xytext=(52, -3),
        arrowprops=dict(
            arrowstyle='-|>', 
            lw=2.2,
            color='black',
            shrinkA=0, shrinkB=0, 
            mutation_scale=25    
        ),
        annotation_clip=False
    )
    ax.text(59.5, -4.6, 'Attack Direction', ha='center', va='top', fontsize=11)

    return ax

# Exemplo de uso
draw_pitch()
plt.show()




# Campo Plotly

In [ ]:
import plotly.graph_objects as go
import numpy as np

def draw_pitch_plotly():
    fig = go.Figure()

    # Contorno do campo
    fig.add_shape(type="rect", x0=0, y0=0, x1=120, y1=80, line=dict(color="black", width=2))

    # Linha de meio campo
    fig.add_shape(type="line", x0=60, y0=0, x1=60, y1=80, line=dict(color="black", width=2))

    # Círculo central
    center_circle = go.Scatter(
        x=[60 + 9.15 * np.cos(t) for t in np.linspace(0, 2 * np.pi, 200)],
        y=[40 + 9.15 * np.sin(t) for t in np.linspace(0, 2 * np.pi, 200)],
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False,
        hoverinfo="skip"
    )
    fig.add_trace(center_circle)

    fig.add_trace(go.Scatter(x=[60], y=[40], mode='markers',
                             marker=dict(color='black', size=6),
                             showlegend=False,
                             hoverinfo="skip"))

    # Áreas grandes
    fig.add_shape(type="rect", x0=0, y0=18, x1=18, y1=62, line=dict(color="black", width=2))
    fig.add_shape(type="rect", x0=102, y0=18, x1=120, y1=62, line=dict(color="black", width=2))

    # Arcos das áreas (esquerda)
    arc_left = []
    for angle in np.radians(np.linspace(310, 410, 200)):
        x = 12 + (18.3 / 2) * np.cos(angle)
        y = 40 + (18.3 / 2) * np.sin(angle)
        arc_left.append((x, y))
    fig.add_trace(go.Scatter(x=[pt[0] for pt in arc_left], y=[pt[1] for pt in arc_left],
                             mode="lines", line=dict(color="black", width=2),
                             showlegend=False, hoverinfo="skip"))

    # Arcos das áreas (direita)
    arc_right = []
    for angle in np.radians(np.linspace(130, 230, 200)):
        x = 108 + (18.3 / 2) * np.cos(angle)
        y = 40 + (18.3 / 2) * np.sin(angle)
        arc_right.append((x, y))
    fig.add_trace(go.Scatter(x=[pt[0] for pt in arc_right], y=[pt[1] for pt in arc_right],
                             mode="lines", line=dict(color="black", width=2),
                             showlegend=False, hoverinfo="skip"))

    # Áreas pequenas
    fig.add_shape(type="rect", x0=0, y0=30, x1=6, y1=50, line=dict(color="black", width=2))
    fig.add_shape(type="rect", x0=114, y0=30, x1=120, y1=50, line=dict(color="black", width=2))

    # Marcas de penálti
    fig.add_trace(go.Scatter(x=[12, 108], y=[40, 40], mode='markers',
                             marker=dict(color='black', size=6),
                             showlegend=False,
                             hoverinfo="skip"))

    # Balizas
    fig.add_shape(type="line", x0=0, y0=36, x1=0, y1=44, line=dict(color="black", width=4))
    fig.add_shape(type="line", x0=120, y0=36, x1=120, y1=44, line=dict(color="black", width=4))

    sx, sy = 52, -3  # início
    ex, ey = 68, -3  # fim

    # Linha da seta
    fig.add_trace(go.Scatter(
        x=[sx, ex], y=[sy, ey],
        mode='lines',
        line=dict(color='black', width=2.2),
        hoverinfo='skip',
        showlegend=False
    ))

    # Ponta (triângulo customizado)
    dx = ex - sx
    dy = ey - sy
    norm = np.sqrt(dx**2 + dy**2)
    ux, uy = dx / norm, dy / norm

    arrow_length = 1.5  # controla o comprimento da seta
    head_base = 0.9     # controla a largura

    tip = (ex, ey)
    left = (ex - arrow_length * ux + head_base * uy,
            ey - arrow_length * uy - head_base * ux)
    right = (ex - arrow_length * ux - head_base * uy,
             ey - arrow_length * uy + head_base * ux)

    fig.add_trace(go.Scatter(
        x=[tip[0], left[0], right[0], tip[0]],
        y=[tip[1], left[1], right[1], tip[1]],
        fill='toself',
        fillcolor='black',
        line=dict(color='black'),
        mode='lines',
        hoverinfo='skip',
        showlegend=False
    ))
    
    fig.add_annotation(
        x=59.5, y=-4,
        text="Attack Direction",
        showarrow=False,
        font=dict(size=12),
        xanchor='center',
        yanchor='top'
    )
    
    # Configurações finais
    fig.update_layout(
        xaxis=dict(range=[0, 122], showgrid=False, zeroline=False, visible=False),
        yaxis=dict(range=[-7, 85], showgrid=False, zeroline=False, visible=False, scaleanchor="x", scaleratio=1),
        plot_bgcolor='white',
        margin=dict(l=0, r=0, t=0, b=0),
        width=1000,
        height=700
    )

    return fig

# Exemplo de uso
fig = draw_pitch_plotly()
fig.show()


# HeatMap (Matplotlib)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import os
from pandas import json_normalize
from matplotlib.patches import Polygon
from matplotlib.colors import Normalize
import matplotlib.cm as cm

player_name = "Florian Wirtz"
DATA_PATH = "./data"
comp_id, season_id = 9, 281

matches_path = os.path.join(DATA_PATH, 'matches', str(comp_id), f'{season_id}.json')
with open(matches_path, encoding='utf-8') as f:
    matches = json.load(f)
match_ids = [m['match_id'] for m in matches]


x_all, y_all = [], []
for match_id in match_ids:
    try:
        events_path = os.path.join(DATA_PATH, 'events', f'{match_id}.json')
        with open(events_path, encoding='utf-8') as f:
            events = json.load(f)

        df = json_normalize(events, sep='_')
        df_player = df[df['player_name'] == player_name]

        for loc in df_player['location'].dropna():
            if isinstance(loc, list) and len(loc) == 2:
                x_all.append(loc[0])
                y_all.append(80 - loc[1])  # superior esquerdo -> inferior esquerdo
    except:
        continue

fig, ax = plt.subplots(figsize=(12, 8))

draw_pitch(ax=ax) 

sns.kdeplot(x=x_all, y=y_all, fill=True, cmap='Reds', bw_adjust=0.7, alpha=0.8, thresh=0.05, ax=ax)


# Legenda
legend_ax = fig.add_axes([0.06, -0.13, 0.25, 0.2])
legend_ax.axis('off')
cmap = plt.get_cmap("Reds")
norm = Normalize(vmin=0, vmax=1)

n_steps = 9 # 9 níveis de cor
hex_width = 0.16 # largura pentagono
side = hex_width / 3  # distância entre os vértices inclinados

for i in range(n_steps):
    x_offset = i * 0.065  # posição de cada hexagono
    color = cmap(i / (n_steps - 1)) # atribui um cor a cada 
    # coordenadas do hexagono
    hex_coords = [
        [x_offset, 0.45],
        [x_offset + side, 0.55],
        [x_offset + 2 * side, 0.55],
        [x_offset + 3 * side, 0.45],
        [x_offset + 2 * side, 0.35],
        [x_offset + side, 0.35],
    ]
    polygon = Polygon(hex_coords, closed=True, color=color)
    legend_ax.add_patch(polygon)

# legenda heatmap
legend_ax.text(-0.04, 0.45, "low", fontsize=11, fontweight='bold', va='center', ha='right')
legend_ax.text(x_offset + 0.22, 0.45, "high", fontsize=11, fontweight='bold', va='center', ha='left')
legend_ax.text(x_offset / 2 + 0.10, 0.24, "frequency", fontsize=10, va='center', ha='center', color='gray')


# Posição mediana do jogador
median_x = np.median(x_all)
median_y = np.median(y_all)
ax.scatter(median_x, median_y, color='#0096c7', edgecolor='black', linewidth=1.5, s=100, zorder=5)

# legenda posição mediana
fig.scatter = fig.add_axes([0.74, -0.09, 0.1, 0.1])
fig.scatter.axis("off")
fig.scatter.scatter(0, 0.5, color='#0096c7', edgecolor='black', linewidth=1.5,s=100)
fig.scatter.text(0.026, 0.499, "Median Event Location", fontsize=11, fontweight='bold', va='center', ha='left', color='black');
plt.savefig("heatmap.png", bbox_inches="tight")

# Passes

### Todos os passes

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pandas import json_normalize

# Parâmetros
player_name = "Florian Wirtz"
DATA_PATH = "./data"
comp_id, season_id = 9, 281

# Lê os jogos
with open(os.path.join(DATA_PATH, 'matches', str(comp_id), f'{season_id}.json'), encoding='utf-8') as f:
    matches = json.load(f)
match_ids = [m['match_id'] for m in matches]

# Agrupar passes
passes_sucesso = []
passes_erro = []

match_dict = {}
for m in matches:
    match_id = m['match_id']
    home = m['home_team']['home_team_name']
    away = m['away_team']['away_team_name']
    match_dict[match_id] = f"{home} vs {away}"

for match_id in match_ids:
    try:
        with open(os.path.join(DATA_PATH, 'events', f'{match_id}.json'), encoding='utf-8') as f:
            events = json.load(f)
        df = json_normalize(events, sep='_')
        df_player = df[(df['player_name'] == player_name) & (df['type_name'] == 'Pass')]

        for _, row in df_player.iterrows():
            start = row.get("location", None)
            end = row.get("pass_end_location", None)

            if not isinstance(start, list) or not isinstance(end, list):
                continue

            start_x, start_y = start[0], 80 - start[1]
            end_x, end_y = end[0], 80 - end[1]
            
            body_part = row.get("pass_body_part_name", 'Unknown')
            match_name = match_dict.get(match_id, f"Match {match_id}")
            height = row.get("pass_height_name", "Unknown")
            length = row.get("pass_length", "N/A")

            hover_text = (
                f"Match: {match_name}<br>"
                f"Body part: {body_part}<br>"
                f"Height: {height}<br>"
                f"Length: {length} meters"
            )

            if pd.isna(row.get("pass_outcome_name", None)):
                passes_sucesso.append(((start_x, start_y), (end_x, end_y), hover_text))
            else:
                passes_erro.append(((start_x, start_y), (end_x, end_y), hover_text))
    except:
        continue

# Função campo
fig = draw_pitch_plotly()

# Desenhar passes
import matplotlib.colors as colors  # importa para converter cores

def add_pass_arrows(pass_list, color, alpha=0.55):
    rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
    
    for (sx, sy), (ex, ey), hover in pass_list:
        fig.add_trace(go.Scatter(
            x=[sx, ex], y=[sy, ey],
            mode='lines',
            line=dict(color=rgba, width=1.5),
            text=[hover, hover],
            hoverinfo='text',
            showlegend=False
        ))

        dx = ex - sx
        dy = ey - sy
        norm = np.sqrt(dx**2 + dy**2)
        if norm == 0:
            continue
        ux, uy = dx / norm, dy / norm
        arrow_length = 0.75
        head_base = 0.42
        tip = (ex, ey)
        left = (ex - arrow_length * ux + head_base * uy, ey - arrow_length * uy - head_base * ux)
        right = (ex - arrow_length * ux - head_base * uy, ey - arrow_length * uy + head_base * ux)

        fig.add_trace(go.Scatter(
            x=[tip[0], left[0], right[0], tip[0]],
            y=[tip[1], left[1], right[1], tip[1]],
            fill='toself',
            fillcolor=rgba,
            line=dict(color=rgba),
            mode='lines',
            hoverinfo='skip',
            showlegend=False
        ))


# Adiciona sucesso e erro
add_pass_arrows(passes_sucesso, '#1F77B4')  
add_pass_arrows(passes_erro, '#D62728')     

fig.update_layout(
    height=800,
    width=1000,
    margin=dict(l=0, r=0, t=0, b=0),
    yaxis=dict(range=[-5, 85], showgrid=False, zeroline=False, visible=False),
    xaxis=dict(showgrid=False, zeroline=False, visible=False)
)

fig.show()


### Passes bem sucedidos

In [ ]:
fig = draw_pitch_plotly()

def add_pass_arrows(pass_list, color, alpha=0.55):
    rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
    
    for (sx, sy), (ex, ey), hover in pass_list:
        fig.add_trace(go.Scatter(
            x=[sx, ex], y=[sy, ey],
            mode='lines',
            line=dict(color=rgba, width=1.5),
            text=[hover, hover],
            hoverinfo='text',
            showlegend=False
        ))

        dx = ex - sx
        dy = ey - sy
        norm = np.sqrt(dx**2 + dy**2)
        if norm == 0:
            continue
        ux, uy = dx / norm, dy / norm
        arrow_length = 0.75
        head_base = 0.42
        tip = (ex, ey)
        left = (ex - arrow_length * ux + head_base * uy, ey - arrow_length * uy - head_base * ux)
        right = (ex - arrow_length * ux - head_base * uy, ey - arrow_length * uy + head_base * ux)

        fig.add_trace(go.Scatter(
            x=[tip[0], left[0], right[0], tip[0]],
            y=[tip[1], left[1], right[1], tip[1]],
            fill='toself',
            fillcolor=rgba,
            line=dict(color=rgba),
            mode='lines',
            hoverinfo='skip',
            showlegend=False
        ))
        
# Adiciona sucesso
add_pass_arrows(passes_sucesso, '#1F77B4') 

fig.show()

### Passes mal sucedidos

In [ ]:
fig = draw_pitch_plotly()

def add_pass_arrows(pass_list, color, alpha=0.55):
    rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
    
    for (sx, sy), (ex, ey), hover in pass_list:
        fig.add_trace(go.Scatter(
            x=[sx, ex], y=[sy, ey],
            mode='lines',
            line=dict(color=rgba, width=1.5),
            text=[hover, hover],
            hoverinfo='text',
            showlegend=False
        ))

        dx = ex - sx
        dy = ey - sy
        norm = np.sqrt(dx**2 + dy**2)
        if norm == 0:
            continue
        ux, uy = dx / norm, dy / norm
        arrow_length = 0.75
        head_base = 0.42
        tip = (ex, ey)
        left = (ex - arrow_length * ux + head_base * uy, ey - arrow_length * uy - head_base * ux)
        right = (ex - arrow_length * ux - head_base * uy, ey - arrow_length * uy + head_base * ux)

        fig.add_trace(go.Scatter(
            x=[tip[0], left[0], right[0], tip[0]],
            y=[tip[1], left[1], right[1], tip[1]],
            fill='toself',
            fillcolor=rgba,
            line=dict(color=rgba),
            mode='lines',
            hoverinfo='skip',
            showlegend=False
        ))

        
# Adiciona erro
add_pass_arrows(passes_erro, '#D62728') 

fig.show()

### Passes chave (bem sucedidos)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pandas import json_normalize
import matplotlib.colors as colors
import matplotlib.colors as mcolors

player_name = "Florian Wirtz"
DATA_PATH = "./data"
comp_id, season_id = 9, 281

with open(os.path.join(DATA_PATH, 'matches', str(comp_id), f'{season_id}.json'), encoding='utf-8') as f:
    matches = json.load(f)
match_ids = [m['match_id'] for m in matches]

# Dicionário de nomes dos jogos
match_dict = {}
for m in matches:
    match_id = m['match_id']
    home = m['home_team']['home_team_name']
    away = m['away_team']['away_team_name']
    match_dict[match_id] = f"{home} vs {away}"

# coletar passes que deram remate e passes que deram golo
passes_chave = []
assistencias = []

for match_id in match_ids:
    try:
        with open(os.path.join(DATA_PATH, 'events', f'{match_id}.json'), encoding='utf-8') as f:
            events = json.load(f)
        df = json_normalize(events, sep='_')
        df_player = df[(df['player_name'] == player_name) & (df['type_name'] == 'Pass')]

        for _, row in df_player.iterrows():
            start = row.get("location")
            end = row.get("pass_end_location")
            if not isinstance(start, list) or not isinstance(end, list):
                continue

            start_x, start_y = start[0], 80 - start[1]
            end_x, end_y = end[0], 80 - end[1]

            recipient = row.get("pass_recipient_name", 'Unknown')
            body_part = row.get("pass_body_part_name", 'Unknown')
            match_name = match_dict.get(match_id, f"Match {match_id}")
            height = row.get("pass_height_name", "Unknown")
            length = row.get("pass_length", "N/A")

            hover_text = (
                f"Match: {match_name}<br>"
                f"Recipient: {recipient}<br>"
                f"Body part: {body_part}<br>"
                f"Height: {height}<br>"
                f"Length: {length} meters"
            )

            if row.get("pass_shot_assist") is True:
                passes_chave.append(((start_x, start_y), (end_x, end_y), hover_text))

            if row.get("pass_goal_assist") is True:
                assistencias.append(((start_x, start_y), (end_x, end_y), hover_text))
    except:
        continue

# Adicionar passes com setas e hover
def add_pass_arrows(fig, pass_list, color, alpha=0.7):
    rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
    
    for (sx, sy), (ex, ey), hover in pass_list:
        fig.add_trace(go.Scatter(
            x=[sx, ex], y=[sy, ey],
            mode='lines',
            line=dict(color=rgba, width=1.5),
            text=[hover, hover],
            hoverinfo='text',
            showlegend=False
        ))

        dx = ex - sx
        dy = ey - sy
        norm = np.sqrt(dx**2 + dy**2)
        if norm == 0:
            continue
        ux, uy = dx / norm, dy / norm
        arrow_length = 0.75
        head_base = 0.42
        tip = (ex, ey)
        left = (ex - arrow_length * ux + head_base * uy, ey - arrow_length * uy - head_base * ux)
        right = (ex - arrow_length * ux - head_base * uy, ey - arrow_length * uy + head_base * ux)

        fig.add_trace(go.Scatter(
            x=[tip[0], left[0], right[0], tip[0]],
            y=[tip[1], left[1], right[1], tip[1]],
            fill='toself',
            fillcolor=rgba,
            line=dict(color=rgba),
            mode='lines',
            hoverinfo='skip',
            showlegend=False
        ))
        
# --- Passes que deram remate ---
fig_remates = draw_pitch_plotly()
add_pass_arrows(fig_remates, passes_chave, '#1F77B4')
fig_remates.show()

### Assistências (bem sucedidos)

In [ ]:
# --- Passes que deram golo (assistências) ---
fig_golos = draw_pitch_plotly()
add_pass_arrows(fig_golos, assistencias, '#1F77B4')
fig_golos.show()


### Gráficos Complementares

In [ ]:
import plotly.express as px

success_count = len(passes_sucesso)
error_count = len(passes_erro)

df_pie = pd.DataFrame({
    'Result': ['Successful', 'Unsuccessful'],
    'Count': [success_count, error_count]
})

fig_pie = px.pie(df_pie,
                 names='Result',
                 values='Count',
                 color='Result',
                 title='Pass Success Rate',
                 color_discrete_map={
                     'Successful': '#1F77B4',   
                     'Unsuccessful': '#D62728' 
                 },
                 hole=0.35)  

# Percentagens dentro
fig_pie.update_traces(textposition='inside', textinfo='percent+label')

# Layout limpo
fig_pie.update_layout(showlegend=False, template='plotly_white')

fig_pie.show()


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import plotly.express as px
from pandas import json_normalize
import plotly.graph_objects as go

''' 
Indicações dataset
 0 = Passe para frente (direção do ataque)
+π (≈ 3.14) = Passe para trás
±π/2 (≈ ±1.57) = Passe lateral

Como defini
Backward: angle > 3π/4 (≈ 2.36) ou angle < -3π/4 (≈ -2.36)
Left (lateral esquerda): -3π/4 a -π/4 (≈ -2.36 a -0.79)
Right (lateral direita): π/4 a 3π/4 (≈ 0.79 a 2.36)
Forward: -π/4 a π/4 (≈ -0.79 a 0.79)
'''

passes = []
for match_id in match_ids:
    try:
        with open(os.path.join(DATA_PATH, 'events', f'{match_id}.json'), encoding='utf-8') as f:
            events = json.load(f)
        df = json_normalize(events, sep='_')
        df_passes = df[(df['player_name'] == player_name) & (df['type_name'] == 'Pass')][['pass_angle', 'pass_outcome_name']].dropna(subset=['pass_angle'])
        passes.append(df_passes)
    except:
        continue

df_passes = pd.concat(passes, ignore_index=True)

def classify_pass_direction(angle):
    if -2*np.pi/3 <= angle <= -np.pi/3 or np.pi/3 <= angle <= 2*np.pi/3:
        return 'Lateral'
    elif -np.pi/3 < angle < np.pi/3:
        return 'Forward'
    else:
        return 'Backward'

df_passes['Direction'] = df_passes['pass_angle'].apply(classify_pass_direction)
df_passes['Result'] = df_passes['pass_outcome_name'].apply(lambda x: 'Successful' if pd.isna(x) else 'Unsuccessful')

counts_dir = df_passes.groupby(['Direction', 'Result']).size().reset_index(name='count')
pivot_dir = counts_dir.pivot(index='Direction', columns='Result', values='count').fillna(0)

colors = {'Successful': '#1F77B4', 'Unsuccessful': '#D62728'}

fig_dir = go.Figure()
for result in ['Unsuccessful', 'Successful']:
    fig_dir.add_trace(go.Bar(
        x=pivot_dir.index,
        y=pivot_dir[result],
        name=result,
        marker_color=colors[result],
        customdata=[[f"{(v / pivot_dir.loc[cat].sum()) * 100:.1f}%"] for cat, v in zip(pivot_dir.index, pivot_dir[result])],
        hovertemplate='%{y} passes<br>%{x}<br>%{customdata[0]}'
    ))

for cat in pivot_dir.index:
    total = pivot_dir.loc[cat].sum()
    fig_dir.add_annotation(
        x=cat,
        y=total + 20,  
        text=f"{int(total)} passes",
        showarrow=False,
        font=dict(size=12, color='black')
    )

fig_dir.update_layout(
    barmode='stack',
    title='Pass Direction',
    xaxis_title='Pass Direction',
    yaxis_title='Number of Passes',
    legend_title='Result',
    bargap=0.2,
    xaxis_title_standoff=20,
    template='plotly_white'
)

fig_dir.show()



In [ ]:
passes_len = []
for match_id in match_ids:
    try:
        with open(os.path.join(DATA_PATH, 'events', f'{match_id}.json'), encoding='utf-8') as f:
            events = json.load(f)
        df = json_normalize(events, sep='_')
        df_passes = df[(df['player_name'] == player_name) & (df['type_name'] == 'Pass')][['pass_length', 'pass_outcome_name']].dropna(subset=['pass_length'])
        passes_len.append(df_passes)
    except:
        continue

df_passes_len = pd.concat(passes_len, ignore_index=True)

def classify_pass_length(length):
    if length <= 10:
        return 'Short (0–10m)'
    elif length <= 25:
        return 'Medium (10–25m)'
    else:
        return 'Long (>25m)'

df_passes_len['Length Category'] = df_passes_len['pass_length'].apply(classify_pass_length)
df_passes_len['Result'] = df_passes_len['pass_outcome_name'].apply(lambda x: 'Successful' if pd.isna(x) else 'Unsuccessful')

counts_len = df_passes_len.groupby(['Length Category', 'Result']).size().reset_index(name='count')
pivot_len = counts_len.pivot(index='Length Category', columns='Result', values='count').fillna(0)
pivot_len = pivot_len.reindex(['Short (0–10m)', 'Medium (10–25m)', 'Long (>25m)'])

fig_len = go.Figure()
for result in ['Unsuccessful', 'Successful']:
    fig_len.add_trace(go.Bar(
        x=pivot_len.index,
        y=pivot_len[result],
        name=result,
        marker_color=colors[result],
        customdata=[[f"{(v / pivot_len.loc[cat].sum()) * 100:.1f}%"] for cat, v in zip(pivot_len.index, pivot_len[result])],
        hovertemplate='%{y} passes<br>%{x}<br>%{customdata[0]}'
    ))

for cat in pivot_len.index:
    total = pivot_len.loc[cat].sum()
    fig_len.add_annotation(
        x=cat,
        y=total + 20,
        text=f"{int(total)} passes",
        showarrow=False,
        font=dict(size=12, color='black')
    )


fig_len.update_layout(
    barmode='stack',
    title='Pass Length',
    xaxis_title='Pass Length Category',
    yaxis_title='Number of Passes',
    legend_title='Result',
    bargap=0.2,
    template='plotly_white'
)

fig_len.show()

# Remates

In [ ]:
import plotly.graph_objects as go
import numpy as np

def draw_vertical_half_pitch():
    fig = go.Figure()

    # Dimensões principais
    pitch_width = 80
    pitch_height = 60
    goal_width = 8
    penalty_box_width = 44
    penalty_box_height = 18
    six_yard_box_width = 20
    six_yard_box_height = 6
    penalty_spot_y = 48
    arc_radius = 9.15

    # Campo
    fig.add_shape(type="rect", x0=0, y0=0, x1=80, y1=60, line=dict(color="black", width=2))

    # Área grande
    fig.add_shape(type="rect", x0=18, y0=42, x1=62, y1=60, line=dict(color="black", width=2))

    # Pequena área
    fig.add_shape(type="rect", x0=30, y0=54, x1=50, y1=60, line=dict(color="black", width=2))

    # Baliza
    fig.add_shape(type="line", x0=36, y0=60, x1=44, y1=60, line=dict(color="black", width=4))

    # Pontos de penalti e centro
    fig.add_trace(go.Scatter(x=[40], y=[0], mode="markers", marker=dict(size=6, color="black"), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[40], y=[48], mode="markers", marker=dict(size=6, color="black"), showlegend=False, hoverinfo="skip"))

    
    arc_angle_start = 310
    arc_angle_end = 50
    angle_step = 1  

    arc_angles = np.linspace(arc_angle_start, 360, num=300, endpoint=False).tolist() + \
             np.linspace(0, arc_angle_end, num=300).tolist()

    # Parâmetros do arco
    center_x = 40
    center_y = 48
    width = 18.3
    height = 18.3
    rotation_deg = 270  # tal como em matplotlib
    rotation_rad = np.radians(rotation_deg)

    arc_x = []
    arc_y = []

    for theta_deg in arc_angles:
        theta = np.radians(theta_deg)
        x0 = (width / 2) * np.cos(theta)
        y0 = (height / 2) * np.sin(theta)

        # Rotação da elipse
        x_rot = x0 * np.cos(rotation_rad) - y0 * np.sin(rotation_rad)
        y_rot = x0 * np.sin(rotation_rad) + y0 * np.cos(rotation_rad)

        arc_x.append(center_x + x_rot)
        arc_y.append(center_y + y_rot)

    fig.add_trace(go.Scatter(
        x=arc_x,
        y=arc_y,
        mode='lines',
        line=dict(color="black", width=2),
        showlegend=False,
        hoverinfo="skip"
    ))

    # --- Meio-campo: semicírculo virado para cima ---
    theta_vals = np.linspace(0, np.pi, 200)
    center_x = 40
    center_y = 0
    radius = 9.15

    mid_arc_x = center_x + radius * np.cos(theta_vals)
    mid_arc_y = center_y + radius * np.sin(theta_vals)

    fig.add_trace(go.Scatter(
        x=mid_arc_x,
        y=mid_arc_y,
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False,
        hoverinfo="skip"
    ))
    
    fig.update_layout(
        xaxis=dict(range=[0, 80], visible=False),
        yaxis=dict(range=[-2, 62], visible=False, scaleanchor="x", scaleratio=1),
        margin=dict(l=0, r=0, t=0, b=0),
        plot_bgcolor="white",
        width=700,
        height=int(800 * (64 / 80))  
    )

    return fig

def plot_shots_with_xg_bubbles(match_ids, player_name, data_path, fig):
    import os, json
    from pandas import json_normalize

    total_goals = 0
    total_xg = 0
    total_shots = 0
    all_shots = []

    # 1. Obter nome dos jogos (match_id -> "Team A vs Team B")
    matches_path = os.path.join(data_path, 'matches', '9', '281.json')  # ajusta conforme necessário
    with open(matches_path, encoding='utf-8') as f:
        matches = json.load(f)

    match_dict = {}
    for m in matches:
        match_id = m['match_id']
        home = m['home_team']['home_team_name']
        away = m['away_team']['away_team_name']
        match_dict[match_id] = f"{home} vs {away}"

    # 2. Coletar todos os remates
    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f'{match_id}.json'), encoding='utf-8') as f:
                events = json.load(f)
            df = json_normalize(events, sep='_')
            shots = df[(df['player_name'] == player_name) & (df['type_name'] == 'Shot')]

            for _, row in shots.iterrows():
                loc = row.get('location')
                xg = float(row.get('shot_statsbomb_xg', 0))
                is_goal = row.get('shot_outcome_name') == 'Goal'
                body_part = row.get('shot_body_part_name', 'Unknown')
                shot_type = row.get('shot_type_name', 'Unknown')
                match_name = match_dict.get(match_id, f"Match {match_id}")

                if not isinstance(loc, list) or len(loc) < 2:
                    continue
                x, y = loc[0], loc[1]
                if x < 60:
                    continue

                total_xg += xg
                total_shots += 1
                if is_goal:
                    total_goals += 1

                x_new = 80 - y
                y_new = x - 60
                color = 'red' if is_goal else 'orange'

                all_shots.append({
                    'x': x_new,
                    'y': y_new,
                    'xg': xg,
                    'color': color,
                    'hover': f"xG: {xg:.2f}<br>{'Goal' if is_goal else 'Shot'}<br>Match: {match_name}<br>Body part: {body_part}<br>Type: {shot_type}"
                })

        except Exception as e:
            print(f"Erro no jogo {match_id}: {e}")
            continue

    # 3. Definir escala das bolhas (área proporcional)
    def xg_to_radius(xg, max_radius=40, min_radius=10):
        return min_radius + (max_radius - min_radius) * (xg ** 0.9)

    for shot in all_shots:
        fig.add_trace(go.Scatter(
            x=[shot['x']],
            y=[shot['y']],
            mode='markers',
            marker=dict(
                size=xg_to_radius(shot['xg']),
                color=shot['color'],
                line=dict(color='black', width=1.2),
                opacity=0.8
            ),
            hoverinfo='text',
            hovertext=shot['hover'],
            showlegend=False
        ))

    # 4. Estatísticas anotadas
    xg_per_shot = total_xg / total_shots if total_shots else 0
    base_y = 5.5
    spacing = 3.8
    stats_annotations = [
        dict(x=4, y=base_y + spacing * 4, text=f"<b>Summary</b>", showarrow=False, font=dict(size=16, color="black", family="Helvetica"), xanchor='left'),
        dict(x=4, y=base_y + spacing * 3, text=f"Goals: <b>{total_goals}</b>", showarrow=False, font=dict(size=16, color="black", family="Helvetica"), xanchor='left'),
        dict(x=4, y=base_y + spacing * 2, text=f"npxG: <b>{total_xg:.2f}</b>", showarrow=False, font=dict(size=16, color="black", family="Helvetica"), xanchor='left'),
        dict(x=4, y=base_y + spacing, text=f"Shots: <b>{total_shots}</b>", showarrow=False, font=dict(size=16, color="black", family="Helvetica"), xanchor='left'),
        dict(x=4, y=base_y, text=f"npxG/Shot: <b>{xg_per_shot:.2f}</b>", showarrow=False, font=dict(size=16, color="black", family="Helvetica"), xanchor='left'),
    ]

    # 5. Legenda e bolinhas de xG
    fig.add_trace(go.Scatter(
        x=[11],
        y=[-3],
        mode="markers",
        marker=dict(size=18, color='orange', line=dict(color='black', width=1.5)),
        showlegend=False,
        hoverinfo="skip"
    ))
    fig.add_trace(go.Scatter(
        x=[18],
        y=[-3],
        mode="markers",
        marker=dict(size=18, color='red', line=dict(color='black', width=1.5)),
        showlegend=False,
        hoverinfo="skip"
    ))

    # 2. Adiciona as anotações restantes com texto
    legend_annotations = [
        dict(x=9, y=-3, text="Outcome:", showarrow=False,
             font=dict(size=14, color="black", family="Helvetica", weight='bold'), xanchor='right'),

        dict(x=14, y=-3, text="Shot", showarrow=False,
             font=dict(size=14, color="black", family="Helvetica")),

        dict(x=21.1, y=-3, text="Goal", showarrow=False,
             font=dict(size=14, color="black", family="Helvetica")),

        dict(x=52, y=-3, text="xG Value:", showarrow=False,
             font=dict(size=14, color="black", family="Helvetica", weight='bold'), xanchor='right')
    ]


    xg_examples = [0.1, 0.4, 0.7, 1.0]
    x_positions = [53.5, 60, 67, 74.9]
    text_positions = [56, 63, 70.5, 78.9]
    for xg, x_pos, text_pos in zip(xg_examples, x_positions, text_positions):
        fig.add_trace(go.Scatter(
            x=[x_pos],
            y=[-3],
            mode='markers',
            marker=dict(
                size=xg_to_radius(xg),
                color='white',
                line=dict(color='black', width=1.2),
                opacity=0.8
            ),
            showlegend=False
        ))
        legend_annotations.append(
            dict(x=text_pos, y=-3, text=f"{xg}", showarrow=False, font=dict(size=14, color="black", family="Helvetica"))
        )

    # 6. Atualizar layout
    fig.update_layout(annotations=stats_annotations + legend_annotations)

    return total_goals, total_xg, total_shots


fig = draw_vertical_half_pitch()
total_goals, total_xg, total_shots = plot_shots_with_xg_bubbles(match_ids, player_name, DATA_PATH, fig)

fig.show()

In [ ]:
import os
import json
import pandas as pd
import plotly.graph_objects as go
from pandas import json_normalize

def create_goal_post_plotly():
    fig = go.Figure()

    # Estrutura principal
    fig.add_shape(type='line', x0=32, y0=0, x1=48, y1=0, line=dict(color='lightgray', width=2))
    fig.add_shape(type='line', x0=36, y0=0, x1=36, y1=2.67, line=dict(color='black'))
    fig.add_shape(type='line', x0=44, y0=0, x1=44, y1=2.67, line=dict(color='black'))
    fig.add_shape(type='line', x0=36, y0=2.67, x1=44, y1=2.67, line=dict(color='black'))

    # Parte traseira
    fig.add_shape(type='line', x0=35.9, y0=0, x1=35.9, y1=2.75, line=dict(color='black'))
    fig.add_shape(type='line', x0=44.1, y0=0, x1=44.1, y1=2.75, line=dict(color='black'))
    fig.add_shape(type='line', x0=35.9, y0=2.75, x1=44.1, y1=2.75, line=dict(color='black'))

    # Linhas de profundidade
    fig.add_shape(type='line', x0=36, y0=2.67, x1=36.3, y1=2.58, line=dict(color='gray'))
    fig.add_shape(type='line', x0=36.3, y0=2.58, x1=36.7, y1=0.8, line=dict(color='gray'))
    fig.add_shape(type='line', x0=36.7, y0=0.8, x1=36, y1=0, line=dict(color='gray'))

    fig.add_shape(type='line', x0=44, y0=2.67, x1=43.7, y1=2.58, line=dict(color='gray'))
    fig.add_shape(type='line', x0=43.7, y0=2.58, x1=43.3, y1=0.8, line=dict(color='gray'))
    fig.add_shape(type='line', x0=43.3, y0=0.8, x1=44, y1=0, line=dict(color='gray'))
    fig.add_shape(type='line', x0=36.7, y0=0.8, x1=43.3, y1=0.8, line=dict(color='gray'))

    fig.add_shape(type='line', x0=26, y0=0, x1=54, y1=0, line=dict(color='lightgray', width=2))

    fig.update_layout(
        width=900,
        height=500,
        xaxis=dict(range=[34.8, 45.3], visible=False, scaleanchor="y", scaleratio=1),
        yaxis=dict(range=[0, 3], visible=False),
        plot_bgcolor='white',
        margin=dict(l=20, r=20, t=40, b=0)
    )

    return fig

def get_match_id_name_dict(matches_path):
    with open(matches_path, encoding='utf-8') as f:
        matches = json.load(f)

    match_dict = {}
    for m in matches:
        match_id = m['match_id']
        home = m['home_team']['home_team_name']
        away = m['away_team']['away_team_name']
        match_dict[match_id] = f"{home} vs {away}"
    return match_dict

def plot_shots_plotly_with_hover(match_ids, player_name, data_path, match_dict):
    fig = create_goal_post_plotly()

    shot_styles = {
        'Goal': {'color': '#2CA02C', 'symbol': 'x'},
        'Off T': {'color': '#9467BD', 'symbol': 'x'},
        'Post': {'color': '#FF7F0E', 'symbol': 'x'},
        'Saved': {'color': '#D62728', 'symbol': 'x'},
        'Saved to Post': {'color': '#1F77B4', 'symbol': 'x'},
    }

    display_names = {
        'Goal': 'Goal',
        'Off T': 'Off Target',
        'Post': 'Post',
        'Saved': 'Saved',
        'Saved to Post': 'Saved to Post'
    }

    for outcome, style in shot_styles.items():
        xs, ys, labels = [], [], []
        for match_id in match_ids:
            try:
                with open(os.path.join(data_path, 'events', f'{match_id}.json'), encoding='utf-8') as f:
                    events = json.load(f)
                df = json_normalize(events, sep='_')
                shots = df[(df['player_name'] == player_name) &
                           (df['type_name'] == 'Shot') &
                           (df['shot_outcome_name'] == outcome)]

                for _, row in shots.iterrows():
                    loc = row.get('shot_end_location')
                    body_part = row.get('shot_body_part_name', 'Unknown')
                    if not isinstance(loc, list) or len(loc) < 3:
                        continue
                    x = loc[1]
                    y = loc[2]
                    match_name = match_dict.get(match_id, f"Match ID {match_id}")
                    xs.append(x)
                    ys.append(y)
                    labels.append(
                        f"<b>{display_names[outcome]}</b><br>{match_name}<br>Body part: {body_part}<br>Height: {y:.2f} meters"
                    )
            except Exception as e:
                print(f"Erro no jogo {match_id}: {e}")
                continue

        fig.add_trace(go.Scatter(
            x=xs,
            y=ys,
            mode='markers',
            marker=dict(color=style['color'], size=8, symbol=style['symbol']),
            name=display_names[outcome],
            hovertext=labels,
            hoverinfo="text"
        ))

    fig.update_layout(
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.99,
            font=dict(size=10)
        ),
        xaxis=dict(fixedrange=False),
        yaxis=dict(range=[1, 2.45], fixedrange=False),  # Zoom ON
        dragmode="zoom",
        margin=dict(t=30, b=0, l=10, r=10),  # b=5 -> mínimo espaço em baixo
        height=400
    )

    fig.show()

player_name = "Florian Wirtz"
DATA_PATH = "./data"
comp_id, season_id = 9, 281

with open(os.path.join(DATA_PATH, 'matches', str(comp_id), f'{season_id}.json'), encoding='utf-8') as f:
    matches = json.load(f)
match_ids = [m['match_id'] for m in matches]
matches_path = os.path.join(DATA_PATH, 'matches', '9', '281.json')  

match_dict = get_match_id_name_dict(matches_path)
plot_shots_plotly_with_hover(match_ids, player_name, DATA_PATH, match_dict)

In [ ]:
import plotly.express as px

def shot_resume(match_ids, player_name, data_path):
    outcome_group = {
        "Goal": "Shots on Target",
        "Saved": "Shots on Target",
        "Saved to Post": "Shots on Target",
        "Post": "Shots off Target",
        "Off T": "Shots off Target"
    }

    outcome_labels = {
        "Goal": "Goal",
        "Saved": "Saved",
        "Saved to Post": "Saved to Post",
        "Post": "Post",
        "Off T": "Off Target"
    }

    data = []

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f'{match_id}.json'), encoding='utf-8') as f:
                events = json.load(f)
            df = json_normalize(events, sep='_')
            shots = df[(df['player_name'] == player_name) & (df['type_name'] == 'Shot')]
            for outcome_raw in shots['shot_outcome_name'].dropna():
                if outcome_raw in outcome_group:
                    group = outcome_group[outcome_raw]
                    label = outcome_labels[outcome_raw]
                    data.append((group, label))
        except Exception as e:
            print(f"Erro no jogo {match_id}: {e}")
            continue

    df_sunburst = pd.DataFrame(data, columns=['Group', 'Outcome'])
    df_counts = df_sunburst.value_counts().reset_index(name='Count')

    # sunburst
    fig = px.sunburst(
        df_counts,
        path=['Group', 'Outcome'],
        values='Count',
        color='Group',
        color_discrete_map={
            "Shots on Target": '#1F77B4',
            "Shots off Target": '#D62728'
        },
        hover_data={'Count': True},
    )

    fig.update_traces(textinfo='label+percent entry')

    fig.show()

player_name = "Florian Wirtz"
DATA_PATH = "./data"
comp_id, season_id = 9, 281
# Lê os jogos
with open(os.path.join(DATA_PATH, 'matches', str(comp_id), f'{season_id}.json'), encoding='utf-8') as f:
    matches = json.load(f)
match_ids = [m['match_id'] for m in matches]
matches_path = os.path.join(DATA_PATH, 'matches', '9', '281.json') 
    
shot_resume(match_ids, player_name, DATA_PATH)


In [ ]:
import plotly.graph_objects as go
import os
import json
from pandas import json_normalize

def shot_resume(match_ids, player_name, data_path):
    shot_styles = {
        'Goal': {'color': '#2CA02C'},
        'Off T': {'color': '#9467BD'},
        'Post': {'color': '#FF7F0E'},
        'Saved': {'color': '#D62728'},
        'Saved to Post': {'color': '#1F77B4'},
    }

    display_names = {
        'Goal': 'Goal',
        'Off T': 'Off Target',
        'Post': 'Post',
        'Saved': 'Saved',
        'Saved to Post': 'Saved to Post'
    }

    shot_counts = {key: 0 for key in shot_styles}

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f'{match_id}.json'), encoding='utf-8') as f:
                events = json.load(f)
            df = json_normalize(events, sep='_')
            shots = df[(df['player_name'] == player_name) & (df['type_name'] == 'Shot')]
            for outcome in shots['shot_outcome_name'].dropna():
                if outcome in shot_counts:
                    shot_counts[outcome] += 1
        except:
            continue

    total = sum(shot_counts.values())
    if total == 0:
        print("error")
        return

    data = [
        (display_names[k], round((shot_counts[k] / total) * 100, 1), shot_styles[k]['color'])
        for k in shot_counts if shot_counts[k] > 0
    ]
    data.sort(key=lambda x: x[1], reverse=True)

    labels = [d[0] for d in data]
    values = [d[1] for d in data]
    colors = [d[2] for d in data]

    fig = go.Figure()
    for label, value, color in zip(labels, values, colors):
        fig.add_trace(go.Bar(
            x=[""],
            y=[value],
            name=label,
            marker_color=color,
            text=f"{value}%",
            textposition="inside",
            insidetextanchor="middle",
            textfont=dict(size=11, color="white"),
            cliponaxis=False,
            hovertemplate=f"{label}: {value}%<extra></extra>"
        ))

    fig.update_layout(
        barmode="stack",
        height=520,
        width=180,
        showlegend=False,
        title=None,
        margin=dict(t=20, b=20, l=20, r=20),
        xaxis=dict(showticklabels=False, fixedrange=True),
        yaxis=dict(title="Percentage (%)", range=[0, 100], fixedrange=True, showgrid=False),
        plot_bgcolor='white'
    )

    fig.show()

plot_shot_outcomes_stacked_bar_percent_clean(match_ids, player_name, DATA_PATH)

# Network Passes

In [ ]:
from statsbombpy import sb
from collections import Counter, defaultdict
import pandas as pd

team_name = "Bayer Leverkusen"

minutes_by_player = defaultdict(float)
position_by_player = defaultdict(Counter)

# converter 'mm:ss' em minutos
def parse_time(t):
    if t is None or t == 'Final Whistle':
        return 90.0 
    mins, secs = map(int, t.split(':'))
    return mins + secs / 60

for match_id in match_ids:
    try:
        lineup = sb.lineups(match_id)[team_name]
        for player in lineup.to_dict(orient="records"):
            for pos in player["positions"]:
                start = parse_time(pos.get("from"))
                end = parse_time(pos.get("to"))
                mins_played = max(0, end - start)

                key = (player["player_id"], player["player_name"])
                minutes_by_player[key] += mins_played

                if "position" in pos:
                    position_by_player[key][pos["position"]] += mins_played
    except Exception as e:
        print(f"Erro no jogo {match_id}: {e}")
        continue

top_11 = sorted(minutes_by_player.items(), key=lambda x: x[1], reverse=True)[:14]

result = []
for (player_id, player_name), total_minutes in top_11:
    most_common_pos = position_by_player[(player_id, player_name)].most_common(1)[0][0]
    result.append({
        "player_id": player_id,
        "player_name": player_name,
        "minutes": round(total_minutes, 1),
        "most_common_position": most_common_pos
    })

df_top11_minutes = pd.DataFrame(result)
print(df_top11_minutes)


In [ ]:
import os
import json
import numpy as np
import plotly.graph_objects as go
import networkx as nx
from pandas import json_normalize
from sklearn.preprocessing import MinMaxScaler
from matplotlib import cm, colors

# Dados
player_names = [
    "Lukáš Hrádecký", "Granit Xhaka", "Alejandro Grimaldo García", "Jonathan Tah",
    "Florian Wirtz", "Jeremie Frimpong", "Jonas Hofmann", "Edmond Fayçal Tapsoba",
    "Exequiel Alejandro Palacios", "Victor Okoh Boniface", "Robert Andrich"
]

custom_positions = {
    "Lukáš Hrádecký": (11, 40),
    "Jonathan Tah": (28, 27),
    "Edmond Fayçal Tapsoba": (28, 53),
    "Alejandro Grimaldo García": (46, 70),
    "Jeremie Frimpong": (46, 10),
    "Robert Andrich": (46, 40),
    "Granit Xhaka": (60, 26),
    "Exequiel Alejandro Palacios": (60, 54),
    "Jonas Hofmann": (82, 19),
    "Florian Wirtz": (82, 61),
    "Victor Okoh Boniface": (93, 40),
}

def process_pass_network(match_ids, data_path):
    pass_counts = {name: 0 for name in player_names}
    pair_passes = {}

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f'{match_id}.json'), encoding='utf-8') as f:
                events = json.load(f)
            df = json_normalize(events, sep='_')
            passes = df[df['type_name'] == 'Pass']
            for _, row in passes.iterrows():
                passer = row.get('player_name')
                recipient = row.get('pass_recipient_name')
                if passer in player_names and recipient in player_names:
                    pair = tuple(sorted([passer, recipient]))
                    pair_passes[pair] = pair_passes.get(pair, 0) + 1
                    pass_counts[passer] += 1
        except Exception as e:
            print(f"Erro no jogo {match_id}: {e}")
            continue

    G = nx.Graph()
    for player in custom_positions:
        G.add_node(player)

    for (p1, p2), count in pair_passes.items():
        G.add_edge(p1, p2, weight=count)

    scaler = MinMaxScaler()
    vals = np.array([pass_counts[n] for n in G.nodes]).reshape(-1, 1)
    vals_scaled = scaler.fit_transform(vals).flatten()

    node_sizes = [100 + (6000 - 100) * val for val in vals_scaled]
    edge_widths = [G[u][v]['weight'] * 0.035 for u, v in G.edges]

    norm = colors.Normalize(vmin=min(pass_counts.values()), vmax=max(pass_counts.values()))
    cmap = cm.Reds
    node_colors = [f"rgba{tuple(int(c*255) for c in cmap(norm(pass_counts[n]))[:3]) + (1.0,)}" for n in G.nodes]

    return G, node_sizes, edge_widths, node_colors, pass_counts, pair_passes

def plot_pass_network_plotly(G, node_sizes, edge_widths, node_colors, pass_counts, pair_passes):
    fig = draw_pitch_plotly()
    positions = custom_positions

    for i, (u, v) in enumerate(G.edges):
        x0, y0 = positions[u]
        x1, y1 = positions[v]
        weight = G[u][v]['weight']

        # Linha visível (sem hover)
        fig.add_trace(go.Scatter(
            x=[x0, x1], y=[y0, y1],
            mode='lines',
            line=dict(width=edge_widths[i], color='gray'),
            opacity=0.4,
            hoverinfo='skip',
            showlegend=False
        ))

        # Ponto invisível no meio da linha com tooltip
        mid_x = (x0 + x1) / 2
        mid_y = (y0 + y1) / 2

        fig.add_trace(go.Scatter(
            x=[mid_x], y=[mid_y],
            mode='markers',
            marker=dict(size=10, color='rgba(0,0,0,0)'),  # invisível
            hoverinfo='text',
            hovertext=f"<b>{u}</b> ↔ <b>{v}</b><br>{weight} passes",
            showlegend=False
        ))


    # Nós com hover
    for i, node in enumerate(G.nodes):
        x, y = positions[node]
        fig.add_trace(go.Scatter(
            x=[x], y=[y], mode='markers+text',
            marker=dict(size=node_sizes[i]**0.5, color=node_colors[i], line=dict(color='black', width=1)),
            text=[node],
            textposition='top center',
            textfont=dict(size=12, color='black'),
            hoverinfo='text',
            hovertext=f"<b>{node}</b><br>{pass_counts[node]} total passes",
            showlegend=False
        ))

    # legenda
    legend_y = -6
    x_start = 2
    x_gap = 6

    legend_sizes = [75, 150, 225, 300, 375, 450, 525]
    legend_vals = np.linspace(0, 1, len(legend_sizes))  # valores normalizados de 0 a 1

    for i, (size, norm_val) in enumerate(zip(legend_sizes, legend_vals)):
        rgba = cmap(norm_val)  # retorna tupla (r, g, b, a) entre 0 e 1
        color = f"rgba({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)}, 1)"
        fig.add_trace(go.Scatter(
            x=[x_start + i * x_gap], y=[legend_y],
            mode='markers',
            marker=dict(size=np.sqrt(size), color=color, line=dict(color='black', width=1)),
            hoverinfo="skip",
            showlegend=False
        ))

    # linhas com espessura variável
    y_lines = legend_y - 7.6
    line_widths = [1, 1.5, 3, 4.5, 5, 6.5, 8]
    for i, w in enumerate(line_widths):
        fig.add_shape(
            type="line",
            x0=x_start + i * x_gap - 2, y0=y_lines,
            x1=x_start + i * x_gap + 2, y1=y_lines,
            line=dict(width=w, color='gray')
        )

    # textos explicativos
    fig.add_annotation(
        x=x_start + 49.5, y=legend_y - 3,
        text="Player made more Passes",
        showarrow=False, font=dict(size=12, color="black", family="Helvetica"), xanchor='center'
    )
    fig.add_annotation(
        x=x_start + 51, y=y_lines - 2,
        text="More Passes Between Players",
        showarrow=False, font=dict(size=12, color="black", family="Helvetica"), xanchor='center'
    )

    # setas horizontais
    fig.add_annotation(
        x=x_start + (len(legend_sizes) - 1) * x_gap + 2, y=legend_y - 3,
        ax=x_start - 2, ay=legend_y - 3,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.3, arrowwidth=1, arrowcolor='black'
    )
    fig.add_annotation(
        x=x_start + (len(line_widths) - 1) * x_gap + 2, y=y_lines - 2,
        ax=x_start - 2, ay=y_lines - 2,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1.3, arrowwidth=1, arrowcolor='black'
    )


    fig.update_layout(
        height=800,
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(range=[-17, 85], showgrid=False, zeroline=False, visible=False),
    )

    return fig

G, node_sizes, edge_widths, node_colors, pass_counts, pair_passes = process_pass_network(match_ids, DATA_PATH)
fig = plot_pass_network_plotly(G, node_sizes, edge_widths, node_colors, pass_counts, pair_passes)
fig.show()


# Dribles

### Todos

In [ ]:
import os
import json
import matplotlib.colors as colors
import plotly.graph_objects as go

def plot_player_dribbles(player_name, data_path, comp_id, season_id):
    with open(os.path.join(data_path, 'matches', str(comp_id), f"{season_id}.json"), encoding='utf-8') as f:
        matches = json.load(f)

    match_ids = [m['match_id'] for m in matches]
    match_dict = {
        m['match_id']: f"{m['home_team']['home_team_name']} vs {m['away_team']['away_team_name']}"
        for m in matches
    }

    complete = []
    incomplete = []
    nutmeg_complete = []
    nutmeg_incomplete = []

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f"{match_id}.json"), encoding='utf-8') as f:
                events = json.load(f)

            for e in events:
                if e.get("player", {}).get("name") != player_name:
                    continue
                if e.get("type", {}).get("name") != "Dribble":
                    continue

                loc = e.get("location")
                if not isinstance(loc, list) or len(loc) < 2:
                    continue

                outcome = e.get("dribble", {}).get("outcome", {}).get("name", "Complete")
                match_name = match_dict.get(match_id, f"Match {match_id}")
                nutmeg = e.get("dribble", {}).get("nutmeg", False)
                overrun = e.get("dribble", {}).get("overrun", False)
                no_touch = e.get("dribble", {}).get("no_touch", False)
                under_pressure = e.get("under_pressure", False)

                hover = (
                    f"Match: {match_name}<br>"
                    f"Outcome: {outcome}<br>"
                    f"Nutmeg: {'Yes' if nutmeg else 'No'}<br>"
                    f"Overrun: {'Yes' if overrun else 'No'}<br>"
                    f"No Touch: {'Yes' if no_touch else 'No'}<br>"
                    f"Under Pressure: {'Yes' if under_pressure else 'No'}"
                )

                point = {'x': loc[0], 'y': loc[1], 'hover': hover}

                if outcome == "Complete":
                    (nutmeg_complete if nutmeg else complete).append(point)
                else:
                    (nutmeg_incomplete if nutmeg else incomplete).append(point)

        except:
            continue

    fig = draw_pitch_plotly()

    def add_dribbles(dribble_list, color, symbol='circle', alpha=0.8, size=8):
        rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
        fig.add_trace(go.Scatter(
            x=[p['x'] for p in dribble_list],
            y=[80 - p['y'] for p in dribble_list],  # inverter Y
            mode='markers',
            marker=dict(color=rgba, size=size, symbol=symbol, line=dict(width=0.4, color='black')),
            text=[p['hover'] for p in dribble_list],
            hoverinfo='text',
            showlegend=False
        ))

    add_dribbles(complete, '#1F77B4')  
    add_dribbles(incomplete, '#D62728')  
    add_dribbles(nutmeg_complete, '#1F77B4', symbol='star', size=12) 
    add_dribbles(nutmeg_incomplete, '#D62728', symbol='star', size=12)  

    fig.update_layout(
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(range=[-7, 85], showgrid=False, zeroline=False, visible=False),
        xaxis=dict(showgrid=False, zeroline=False, visible=False)
    )

    fig.show()

plot_player_dribbles(player_name="Florian Wirtz", data_path="./data", comp_id=9, season_id=281)

### Bem sucedidos

In [ ]:
import os
import json
import matplotlib.colors as colors
import plotly.graph_objects as go

def plot_player_dribbles(player_name, data_path, comp_id, season_id):
    with open(os.path.join(data_path, 'matches', str(comp_id), f"{season_id}.json"), encoding='utf-8') as f:
        matches = json.load(f)

    match_ids = [m['match_id'] for m in matches]
    match_dict = {
        m['match_id']: f"{m['home_team']['home_team_name']} vs {m['away_team']['away_team_name']}"
        for m in matches
    }

    complete = []
    incomplete = []
    nutmeg_complete = []
    nutmeg_incomplete = []

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f"{match_id}.json"), encoding='utf-8') as f:
                events = json.load(f)

            for e in events:
                if e.get("player", {}).get("name") != player_name:
                    continue
                if e.get("type", {}).get("name") != "Dribble":
                    continue

                loc = e.get("location")
                if not isinstance(loc, list) or len(loc) < 2:
                    continue

                outcome = e.get("dribble", {}).get("outcome", {}).get("name", "Complete")
                match_name = match_dict.get(match_id, f"Match {match_id}")
                nutmeg = e.get("dribble", {}).get("nutmeg", False)
                overrun = e.get("dribble", {}).get("overrun", False)
                no_touch = e.get("dribble", {}).get("no_touch", False)
                under_pressure = e.get("under_pressure", False)

                hover = (
                    f"Match: {match_name}<br>"
                    f"Outcome: {outcome}<br>"
                    f"Nutmeg: {'Yes' if nutmeg else 'No'}<br>"
                    f"Overrun: {'Yes' if overrun else 'No'}<br>"
                    f"No Touch: {'Yes' if no_touch else 'No'}<br>"
                    f"Under Pressure: {'Yes' if under_pressure else 'No'}"
                )

                point = {'x': loc[0], 'y': loc[1], 'hover': hover}

                if outcome == "Complete":
                    (nutmeg_complete if nutmeg else complete).append(point)
                else:
                    (nutmeg_incomplete if nutmeg else incomplete).append(point)

        except:
            continue

    fig = draw_pitch_plotly()

    def add_dribbles(dribble_list, color, symbol='circle', alpha=0.8, size=8):
        rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
        fig.add_trace(go.Scatter(
            x=[p['x'] for p in dribble_list],
            y=[80 - p['y'] for p in dribble_list],  # inverter Y
            mode='markers',
            marker=dict(color=rgba, size=size, symbol=symbol, line=dict(width=0.4, color='black')),
            text=[p['hover'] for p in dribble_list],
            hoverinfo='text',
            showlegend=False
        ))

    add_dribbles(complete, '#1F77B4')   
    add_dribbles(nutmeg_complete, '#1F77B4', symbol='star', size=12)   

    fig.update_layout(
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(range=[-7, 85], showgrid=False, zeroline=False, visible=False),
        xaxis=dict(showgrid=False, zeroline=False, visible=False)
    )

    fig.show()

plot_player_dribbles(player_name="Florian Wirtz", data_path="./data", comp_id=9, season_id=281)

### Mal sucedidos

In [ ]:
import os
import json
import matplotlib.colors as colors
import plotly.graph_objects as go

def plot_player_dribbles(player_name, data_path, comp_id, season_id):
    # 1. Carregar partidas
    with open(os.path.join(data_path, 'matches', str(comp_id), f"{season_id}.json"), encoding='utf-8') as f:
        matches = json.load(f)

    match_ids = [m['match_id'] for m in matches]
    match_dict = {
        m['match_id']: f"{m['home_team']['home_team_name']} vs {m['away_team']['away_team_name']}"
        for m in matches
    }

    complete = []
    incomplete = []
    nutmeg_complete = []
    nutmeg_incomplete = []

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f"{match_id}.json"), encoding='utf-8') as f:
                events = json.load(f)

            for e in events:
                if e.get("player", {}).get("name") != player_name:
                    continue
                if e.get("type", {}).get("name") != "Dribble":
                    continue

                loc = e.get("location")
                if not isinstance(loc, list) or len(loc) < 2:
                    continue

                outcome = e.get("dribble", {}).get("outcome", {}).get("name", "Complete")
                match_name = match_dict.get(match_id, f"Match {match_id}")
                nutmeg = e.get("dribble", {}).get("nutmeg", False)
                overrun = e.get("dribble", {}).get("overrun", False)
                no_touch = e.get("dribble", {}).get("no_touch", False)
                under_pressure = e.get("under_pressure", False)

                hover = (
                    f"Match: {match_name}<br>"
                    f"Outcome: {outcome}<br>"
                    f"Nutmeg: {'Yes' if nutmeg else 'No'}<br>"
                    f"Overrun: {'Yes' if overrun else 'No'}<br>"
                    f"No Touch: {'Yes' if no_touch else 'No'}<br>"
                    f"Under Pressure: {'Yes' if under_pressure else 'No'}"
                )

                point = {'x': loc[0], 'y': loc[1], 'hover': hover}

                if outcome == "Complete":
                    (nutmeg_complete if nutmeg else complete).append(point)
                else:
                    (nutmeg_incomplete if nutmeg else incomplete).append(point)

        except:
            continue

    fig = draw_pitch_plotly()

    def add_dribbles(dribble_list, color, symbol='circle', alpha=0.8, size=8):
        rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
        fig.add_trace(go.Scatter(
            x=[p['x'] for p in dribble_list],
            y=[80 - p['y'] for p in dribble_list],  # inverter Y
            mode='markers',
            marker=dict(color=rgba, size=size, symbol=symbol, line=dict(width=0.4, color='black')),
            text=[p['hover'] for p in dribble_list],
            hoverinfo='text',
            showlegend=False
        ))

    add_dribbles(incomplete, '#D62728') 
    add_dribbles(nutmeg_incomplete, '#D62728', symbol='star', size=12)  

    fig.update_layout(
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(range=[-7, 85], showgrid=False, zeroline=False, visible=False),
        xaxis=dict(showgrid=False, zeroline=False, visible=False)
    )

    fig.show()

plot_player_dribbles(player_name="Florian Wirtz", data_path="./data", comp_id=9, season_id=281)

### Nutmeg Bem sucedido

In [ ]:
import os
import json
import matplotlib.colors as colors
import plotly.graph_objects as go

def plot_player_dribbles(player_name, data_path, comp_id, season_id):
    with open(os.path.join(data_path, 'matches', str(comp_id), f"{season_id}.json"), encoding='utf-8') as f:
        matches = json.load(f)

    match_ids = [m['match_id'] for m in matches]
    match_dict = {
        m['match_id']: f"{m['home_team']['home_team_name']} vs {m['away_team']['away_team_name']}"
        for m in matches
    }

    complete = []
    incomplete = []
    nutmeg_complete = []
    nutmeg_incomplete = []

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f"{match_id}.json"), encoding='utf-8') as f:
                events = json.load(f)

            for e in events:
                if e.get("player", {}).get("name") != player_name:
                    continue
                if e.get("type", {}).get("name") != "Dribble":
                    continue

                loc = e.get("location")
                if not isinstance(loc, list) or len(loc) < 2:
                    continue

                outcome = e.get("dribble", {}).get("outcome", {}).get("name", "Complete")
                match_name = match_dict.get(match_id, f"Match {match_id}")
                nutmeg = e.get("dribble", {}).get("nutmeg", False)
                overrun = e.get("dribble", {}).get("overrun", False)
                no_touch = e.get("dribble", {}).get("no_touch", False)
                under_pressure = e.get("under_pressure", False)

                hover = (
                    f"Match: {match_name}<br>"
                    f"Outcome: {outcome}<br>"
                    f"Nutmeg: {'Yes' if nutmeg else 'No'}<br>"
                    f"Overrun: {'Yes' if overrun else 'No'}<br>"
                    f"No Touch: {'Yes' if no_touch else 'No'}<br>"
                    f"Under Pressure: {'Yes' if under_pressure else 'No'}"
                )

                point = {'x': loc[0], 'y': loc[1], 'hover': hover}

                if outcome == "Complete":
                    (nutmeg_complete if nutmeg else complete).append(point)
                else:
                    (nutmeg_incomplete if nutmeg else incomplete).append(point)

        except:
            continue

    fig = draw_pitch_plotly()

    def add_dribbles(dribble_list, color, symbol='circle', alpha=0.8, size=8):
        rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
        fig.add_trace(go.Scatter(
            x=[p['x'] for p in dribble_list],
            y=[80 - p['y'] for p in dribble_list],  # inverter Y
            mode='markers',
            marker=dict(color=rgba, size=size, symbol=symbol, line=dict(width=0.4, color='black')),
            text=[p['hover'] for p in dribble_list],
            hoverinfo='text',
            showlegend=False
        ))

    add_dribbles(nutmeg_complete, '#1F77B4', symbol='star', size=12) 

    fig.update_layout(
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(range=[-7, 85], showgrid=False, zeroline=False, visible=False),
        xaxis=dict(showgrid=False, zeroline=False, visible=False)
    )

    fig.show()

plot_player_dribbles(player_name="Florian Wirtz", data_path="./data", comp_id=9, season_id=281)

## Nutmeg mal sucedido

In [ ]:
import os
import json
import matplotlib.colors as colors
import plotly.graph_objects as go

def plot_player_dribbles(player_name, data_path, comp_id, season_id):
    with open(os.path.join(data_path, 'matches', str(comp_id), f"{season_id}.json"), encoding='utf-8') as f:
        matches = json.load(f)

    match_ids = [m['match_id'] for m in matches]
    match_dict = {
        m['match_id']: f"{m['home_team']['home_team_name']} vs {m['away_team']['away_team_name']}"
        for m in matches
    }

    complete = []
    incomplete = []
    nutmeg_complete = []
    nutmeg_incomplete = []

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f"{match_id}.json"), encoding='utf-8') as f:
                events = json.load(f)

            for e in events:
                if e.get("player", {}).get("name") != player_name:
                    continue
                if e.get("type", {}).get("name") != "Dribble":
                    continue

                loc = e.get("location")
                if not isinstance(loc, list) or len(loc) < 2:
                    continue

                outcome = e.get("dribble", {}).get("outcome", {}).get("name", "Complete")
                match_name = match_dict.get(match_id, f"Match {match_id}")
                nutmeg = e.get("dribble", {}).get("nutmeg", False)
                overrun = e.get("dribble", {}).get("overrun", False)
                no_touch = e.get("dribble", {}).get("no_touch", False)
                under_pressure = e.get("under_pressure", False)

                hover = (
                    f"Match: {match_name}<br>"
                    f"Outcome: {outcome}<br>"
                    f"Nutmeg: {'Yes' if nutmeg else 'No'}<br>"
                    f"Overrun: {'Yes' if overrun else 'No'}<br>"
                    f"No Touch: {'Yes' if no_touch else 'No'}<br>"
                    f"Under Pressure: {'Yes' if under_pressure else 'No'}"
                )

                point = {'x': loc[0], 'y': loc[1], 'hover': hover}

                if outcome == "Complete":
                    (nutmeg_complete if nutmeg else complete).append(point)
                else:
                    (nutmeg_incomplete if nutmeg else incomplete).append(point)

        except:
            continue

    fig = draw_pitch_plotly()

    def add_dribbles(dribble_list, color, symbol='circle', alpha=0.8, size=8):
        rgba = f"rgba{(*colors.to_rgba(color)[:3], alpha)}"
        fig.add_trace(go.Scatter(
            x=[p['x'] for p in dribble_list],
            y=[80 - p['y'] for p in dribble_list],  # inverter Y
            mode='markers',
            marker=dict(color=rgba, size=size, symbol=symbol, line=dict(width=0.4, color='black')),
            text=[p['hover'] for p in dribble_list],
            hoverinfo='text',
            showlegend=False
        ))

    add_dribbles(nutmeg_incomplete, '#D62728', symbol='star', size=12)  # laranja

    fig.update_layout(
        margin=dict(l=0, r=0, t=0, b=0),
        yaxis=dict(range=[-7, 85], showgrid=False, zeroline=False, visible=False),
        xaxis=dict(showgrid=False, zeroline=False, visible=False)
    )

    fig.show()

plot_player_dribbles(player_name="Florian Wirtz", data_path="./data", comp_id=9, season_id=281)

In [ ]:
import plotly.express as px

def plot_dribble_success_donut(player_name, comp_id, season_id, data_path):
    successful = 0
    failed = 0

    with open(os.path.join(data_path, 'matches', str(comp_id), f"{season_id}.json"), encoding='utf-8') as f:
        matches = json.load(f)

    match_ids = [m['match_id'] for m in matches]

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f"{match_id}.json"), encoding='utf-8') as f:
                events = json.load(f)

            for e in events:
                if e.get("player", {}).get("name") != player_name:
                    continue
                if e.get("type", {}).get("name") != "Dribble":
                    continue

                outcome = e.get("dribble", {}).get("outcome", {}).get("name", "Complete")
                if outcome == "Complete":
                    successful += 1
                else:
                    failed += 1

        except:
            continue

    labels = ['Successful', 'Failed']
    values = [successful, failed]


    fig = px.pie(
        names=labels,
        values=values,
        title='Dribble Success Rate',
        color=labels,
        color_discrete_map={'Successful': '#1F77B4', 'Failed': '#D62728'},
        hole=0.35
    )
    
    fig.update_traces(
        textposition='inside',
        textinfo='percent',
        textfont_size=10,
        hovertemplate='%{label}<br>%{value} dribbles<br>%{percent}'
    )

    fig.update_traces(textinfo='percent+label')
    fig.update_layout(showlegend=False, template="plotly_white")
    return fig

donut_fig = plot_dribble_success_donut("Florian Wirtz", 9, 281, "./data")
donut_fig.show()

In [ ]:
import os
import json
import plotly.express as px

def plot_nutmeg_success_donut(player_name, comp_id, season_id, data_path):
    nutmeg_successful = 0
    nutmeg_failed = 0


    with open(os.path.join(data_path, 'matches', str(comp_id), f"{season_id}.json"), encoding='utf-8') as f:
        matches = json.load(f)

    match_ids = [m['match_id'] for m in matches]

    for match_id in match_ids:
        try:
            with open(os.path.join(data_path, 'events', f"{match_id}.json"), encoding='utf-8') as f:
                events = json.load(f)

            for e in events:
                if e.get("player", {}).get("name") != player_name:
                    continue
                if e.get("type", {}).get("name") != "Dribble":
                    continue
                if not e.get("dribble", {}).get("nutmeg", False):
                    continue  # só queremos nutmegs

                outcome = e.get("dribble", {}).get("outcome", {}).get("name", "Complete")
                if outcome == "Complete":
                    nutmeg_successful += 1
                else:
                    nutmeg_failed += 1

        except:
            continue

    if nutmeg_successful + nutmeg_failed == 0:
        print("Nenhum drible tipo nutmeg encontrado.")
        return

    labels = ['Successful Nutmeg', 'Failed Nutmeg']
    values = [nutmeg_successful, nutmeg_failed]

    fig = px.pie(
        names=labels,
        values=values,
        title='Nutmeg Dribble Success Rate',
        color=labels,
        color_discrete_map={
            'Successful Nutmeg': '#1F77B4',
            'Failed Nutmeg': '#D62728'
        },
        hole=0.35
    )
    
    fig.update_traces(
        textposition='inside',
        textinfo='percent',
        textfont_size=10,
        hovertemplate='%{label}<br>%{value} dribbles<br>%{percent}'
    )

    fig.update_traces(textinfo='percent+label')
    fig.update_layout(showlegend=False, template="plotly_white")
    return fig
plot_nutmeg_success_donut("Florian Wirtz", 9, 281, "./data")